In [3]:
import duckdb
con = duckdb.connect("../data/yelp.duckdb")

In [5]:
# 1. one row per restaurant
restaurants = con.sql("""
SELECT business_id, name, city, state, latitude, longitude, stars, review_count,
       CASE WHEN is_open = 1 THEN 'Open' ELSE 'Closed' END AS status,
       TRY_CAST(attributes.RestaurantsPriceRange2 AS INTEGER) AS price_range,
       COUNT(*) OVER (PARTITION BY name, state) >= 3 AS is_chain
FROM business_restaurants
""").df()
restaurants.to_csv("../data/exports/restaurants.csv", index=False)

# 2. one row per restaurant and category
cats = con.sql("""
SELECT business_id, TRIM(category) AS category
FROM (SELECT business_id, UNNEST(string_split(categories, ',')) AS category
      FROM business_restaurants)
WHERE TRIM(category) NOT IN ('Restaurants', 'Food', 'Nightlife', 'Bars')
""").df()
cats.to_csv("../data/exports/restaurant_categories.csv", index=False)

# 3. reviews per year and state (already summed up)
by_year = con.sql("""
SELECT YEAR(CAST(r.date AS TIMESTAMP)) AS year, b.state,
       COUNT(*) AS reviews, ROUND(AVG(r.stars), 2) AS avg_stars
FROM review_stats r
JOIN business_restaurants b USING (business_id)
GROUP BY 1, 2
""").df()
by_year.to_csv("../data/exports/reviews_by_year.csv", index=False)